# 20. Generator Fingerprint Visualization — PCA / UMAP

18-B와 19에서 strict balanced 조건에서도 12-way generator attribution이 높게 나왔다.

이번 단계에서는 **왜 분류가 가능한지 representation space에서 시각화**한다.

비교:
- Handcrafted 266-D track representation
- Frozen MERT-v1-95M Layer 10, 768-D track representation

원칙:
- 18-B의 strict balanced 1,428 tracks 사용
- `original_audio × generator = 정확히 1 track`
- Train에서 StandardScaler / PCA / UMAP fit
- Test 228 tracks만 transform해 시각화
- Generator silhouette와 Genre silhouette 비교


## 0. UMAP 설치

`umap-learn`이 없다면 새 **주피터 코드 셀**에서 한 번만 실행:

```python
%pip install -q umap-learn
```

설치 후 커널을 재시작한다.


## 1. 라이브러리 / 경로

In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

try:
    import umap
except ImportError:
    raise ImportError(
        "umap-learn이 없습니다. 새 주피터 코드 셀에서 "
        "`%pip install -q umap-learn` 실행 후 커널을 재시작하세요."
    )

PROJECT_ROOT = Path(
    "/Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project"
)

SEGMENT_PATH = (
    PROJECT_ROOT / "data/metadata/segment_manifest_10s.csv"
)
FEATURE_PATH = (
    PROJECT_ROOT / "data/processed/features/handcrafted_features_10s.csv"
)
MERT_PATH = (
    PROJECT_ROOT / "data/processed/mert/mert95m_v2_layers_float16.npy"
)
STRICT_TRACKS_PATH = (
    PROJECT_ROOT
    / "results/generator_attribution/balanced_controlled/strict_selected_tracks.csv"
)

RESULT_DIR = (
    PROJECT_ROOT / "results/generator_fingerprint_visualization"
)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

BEST_MERT_LAYER = 10
RANDOM_STATE = 42

segments = pd.read_csv(SEGMENT_PATH).reset_index(drop=True)
features = pd.read_csv(FEATURE_PATH)
mert_cache = np.load(MERT_PATH, mmap_mode="r")
strict_tracks = pd.read_csv(STRICT_TRACKS_PATH)

print("Segments      :", len(segments))
print("Features      :", len(features))
print("MERT cache    :", mert_cache.shape)
print("Strict tracks :", len(strict_tracks))

assert len(segments) == 10077
assert len(features) == 10077
assert mert_cache.shape == (10077, 13, 768)
assert len(strict_tracks) == 1428

print("Input QC PASS: True")


## 1-B. Generator 고정 색상표

12개 generator에 서로 다른 고정 색상을 지정합니다.
PCA와 UMAP에서 같은 generator는 항상 같은 색으로 표시됩니다.


In [ ]:

GENERATOR_COLORS = {
    "acestep":     "#1f77b4",
    "audioldm":    "#ff7f0e",
    "brev":        "#2ca02c",
    "diffrhythm":  "#d62728",
    "elevenlabs":  "#9467bd",
    "mubert":      "#8c564b",
    "musicgen":    "#e377c2",
    "producer":    "#7f7f7f",
    "songgen":     "#bcbd22",
    "stableaudio": "#17becf",
    "suno":        "#393b79",
    "udio":        "#e6550d",
}

print("Generator colors:", len(GENERATOR_COLORS))
assert len(GENERATOR_COLORS) == 12


## 2. Handcrafted strict track representation

In [ ]:

segment_meta_cols = set(segments.columns)

extra_meta_cols = {
    "Unnamed: 0",
    "index",
    "cache_index",
    "extraction_success",
    "success",
    "error",
}

feature_cols = [
    c for c in features.columns
    if c != "segment_id"
    and c not in segment_meta_cols
    and c not in extra_meta_cols
    and not str(c).startswith("Unnamed:")
    and pd.api.types.is_numeric_dtype(features[c])
]

assert len(feature_cols) == 266

meta = segments[
    [
        "segment_id",
        "track_sample_id",
        "original_audio",
        "generator",
        "genre",
        "split",
        "label",
    ]
].copy()

merged = meta.merge(
    features[["segment_id"] + feature_cols],
    on="segment_id",
    how="inner",
    validate="one_to_one",
)

selected_ids = set(strict_tracks["track_sample_id"])

strict_seg = merged[
    merged["track_sample_id"].isin(selected_ids)
].copy()

hand_track = (
    strict_seg
    .groupby("track_sample_id", as_index=False)
    .agg(
        {
            **{
                "original_audio": "first",
                "generator": "first",
                "genre": "first",
                "split": "first",
            },
            **{c: "mean" for c in feature_cols},
        }
    )
)

print("Handcrafted track rows:", len(hand_track))
print("Feature dims:", len(feature_cols))

assert len(hand_track) == 1428
assert not hand_track[feature_cols].isna().any().any()

print("Handcrafted Track QC PASS: True")


## 3. MERT Layer 10 strict track representation

In [ ]:

segments = segments.copy()
segments["cache_index"] = np.arange(len(segments))

strict_mert_seg = segments[
    segments["track_sample_id"].isin(selected_ids)
].copy()

mert_rows = []

for track_id, group in strict_mert_seg.groupby("track_sample_id"):
    idx = group["cache_index"].to_numpy(dtype=int)

    emb = np.asarray(
        mert_cache[idx, BEST_MERT_LAYER, :],
        dtype=np.float32,
    ).mean(axis=0)

    first = group.iloc[0]

    mert_rows.append({
        "track_sample_id": track_id,
        "original_audio": first["original_audio"],
        "generator": first["generator"],
        "genre": first["genre"],
        "split": first["split"],
        "embedding": emb,
    })

mert_track = pd.DataFrame(mert_rows)

print("MERT track rows:", len(mert_track))
print("MERT dims:", len(mert_track.iloc[0]["embedding"]))
print("Layer:", BEST_MERT_LAYER)

assert len(mert_track) == 1428
assert len(mert_track.iloc[0]["embedding"]) == 768

print("MERT Track QC PASS: True")


## 4. Strict balance 재확인

In [ ]:

for name, df in [
    ("Handcrafted", hand_track),
    ("MERT", mert_track),
]:
    print("\n", name)

    table = pd.crosstab(
        df["generator"],
        df["split"],
    )

    display(table)

    assert table["train"].nunique() == 1
    assert table["val"].nunique() == 1
    assert table["test"].nunique() == 1
    assert int(table["train"].iloc[0]) == 83
    assert int(table["val"].iloc[0]) == 17
    assert int(table["test"].iloc[0]) == 19

print("Strict Balance QC PASS: True")


## 5. PCA — Train fit / Test transform

In [ ]:

def get_representation(name):
    if name == "Handcrafted":
        train = hand_track[hand_track["split"] == "train"].reset_index(drop=True)
        test = hand_track[hand_track["split"] == "test"].reset_index(drop=True)

        X_train = train[feature_cols].to_numpy(dtype=np.float32)
        X_test = test[feature_cols].to_numpy(dtype=np.float32)

    elif name == "MERT":
        train = mert_track[mert_track["split"] == "train"].reset_index(drop=True)
        test = mert_track[mert_track["split"] == "test"].reset_index(drop=True)

        X_train = np.stack(train["embedding"].to_numpy()).astype(np.float32)
        X_test = np.stack(test["embedding"].to_numpy()).astype(np.float32)

    else:
        raise ValueError(name)

    return train, test, X_train, X_test


pca_results = {}

for name in ["Handcrafted", "MERT"]:
    train, test, X_train, X_test = get_representation(name)

    scaler = StandardScaler()

    X_train_z = scaler.fit_transform(X_train)
    X_test_z = scaler.transform(X_test)

    pca = PCA(
        n_components=2,
        random_state=RANDOM_STATE,
    )

    pca.fit(X_train_z)

    coords = pca.transform(X_test_z)

    pca_results[name] = {
        "train": train,
        "test": test,
        "X_test_z": X_test_z,
        "coords": coords,
        "explained": pca.explained_variance_ratio_,
    }

    print(
        name,
        "explained variance:",
        np.round(pca.explained_variance_ratio_, 4),
        "sum=",
        round(float(pca.explained_variance_ratio_.sum()), 4),
    )


## 6. PCA Test visualization

In [ ]:

def plot_2d(coords, labels, title, save_path):
    fig, ax = plt.subplots(figsize=(10, 8))

    classes = sorted(pd.Series(labels).unique())

    for cls in classes:
        mask = np.asarray(labels) == cls

        ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            label=cls,
            color=GENERATOR_COLORS.get(cls, "#000000"),
            alpha=0.78,
            s=38,
            edgecolors="none",
        )

    ax.set_xlabel("Dimension 1")
    ax.set_ylabel("Dimension 2")
    ax.set_title(title)

    ax.legend(
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=8,
        frameon=True,
    )

    fig.tight_layout()
    fig.savefig(save_path, dpi=180, bbox_inches="tight")
    plt.show()


for name in ["Handcrafted", "MERT"]:
    r = pca_results[name]

    plot_2d(
        r["coords"],
        r["test"]["generator"],
        f"{name} — Strict Balanced Test PCA",
        RESULT_DIR / f"pca_{name.lower()}_strict_test.png",
    )


## 7. UMAP — Train fit / Test transform

In [ ]:

umap_results = {}

for name in ["Handcrafted", "MERT"]:
    train, test, X_train, X_test = get_representation(name)

    scaler = StandardScaler()

    X_train_z = scaler.fit_transform(X_train)
    X_test_z = scaler.transform(X_test)

    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=30,
        min_dist=0.15,
        metric="euclidean",
        random_state=RANDOM_STATE,
        transform_seed=RANDOM_STATE,
    )

    reducer.fit(X_train_z)
    coords = reducer.transform(X_test_z)

    umap_results[name] = {
        "train": train,
        "test": test,
        "X_test_z": X_test_z,
        "coords": coords,
    }

    print(name, "UMAP test shape:", coords.shape)

print("UMAP Transform PASS: True")


## 8. UMAP Test visualization

In [ ]:

for name in ["Handcrafted", "MERT"]:
    r = umap_results[name]

    plot_2d(
        r["coords"],
        r["test"]["generator"],
        f"{name} — Strict Balanced Test UMAP",
        RESULT_DIR / f"umap_{name.lower()}_strict_test.png",
    )


## 9. Generator silhouette score

In [ ]:

rows = []

for name in ["Handcrafted", "MERT"]:
    labels = pca_results[name]["test"]["generator"].to_numpy()

    spaces = {
        "standardized_original": pca_results[name]["X_test_z"],
        "pca_2d": pca_results[name]["coords"],
        "umap_2d": umap_results[name]["coords"],
    }

    for space_name, X in spaces.items():
        rows.append({
            "representation": name,
            "space": space_name,
            "generator_silhouette": silhouette_score(
                X,
                labels,
                metric="euclidean",
            ),
        })

silhouette_df = pd.DataFrame(rows)

display(silhouette_df.round(4))


## 10. Generator vs Genre separability

In [ ]:

rows = []

for name in ["Handcrafted", "MERT"]:
    test = pca_results[name]["test"]
    X = pca_results[name]["X_test_z"]

    generator_s = silhouette_score(
        X,
        test["generator"].to_numpy(),
    )

    genre_s = silhouette_score(
        X,
        test["genre"].to_numpy(),
    )

    rows.append({
        "representation": name,
        "generator_silhouette": generator_s,
        "genre_silhouette": genre_s,
        "generator_minus_genre": generator_s - genre_s,
    })

generator_vs_genre = pd.DataFrame(rows)

display(generator_vs_genre.round(4))


## 11. 저장 / 최종 QC

In [ ]:

silhouette_df.to_csv(
    RESULT_DIR / "generator_silhouette_scores.csv",
    index=False,
    encoding="utf-8-sig",
)

generator_vs_genre.to_csv(
    RESULT_DIR / "generator_vs_genre_silhouette.csv",
    index=False,
    encoding="utf-8-sig",
)

for name in ["Handcrafted", "MERT"]:
    for method, results in [
        ("pca", pca_results),
        ("umap", umap_results),
    ]:
        out_df = results[name]["test"][
            [
                "track_sample_id",
                "original_audio",
                "generator",
                "genre",
                "split",
            ]
        ].copy()

        out_df["x"] = results[name]["coords"][:, 0]
        out_df["y"] = results[name]["coords"][:, 1]

        out_df.to_csv(
            RESULT_DIR / f"{method}_{name.lower()}_strict_test.csv",
            index=False,
            encoding="utf-8-sig",
        )

qc = pd.DataFrame({
    "check": [
        "strict_tracks",
        "handcrafted_dims",
        "mert_dims",
        "best_mert_layer",
        "hand_test_rows",
        "mert_test_rows",
        "silhouette_rows",
        "pca_pngs",
        "umap_pngs",
    ],
    "value": [
        len(strict_tracks),
        len(feature_cols),
        768,
        BEST_MERT_LAYER,
        len(pca_results["Handcrafted"]["test"]),
        len(pca_results["MERT"]["test"]),
        len(silhouette_df),
        (
            RESULT_DIR / "pca_handcrafted_strict_test.png"
        ).exists()
        and (
            RESULT_DIR / "pca_mert_strict_test.png"
        ).exists(),
        (
            RESULT_DIR / "umap_handcrafted_strict_test.png"
        ).exists()
        and (
            RESULT_DIR / "umap_mert_strict_test.png"
        ).exists(),
    ],
})

display(qc)

core_qc_pass = (
    len(strict_tracks) == 1428
    and len(feature_cols) == 266
    and BEST_MERT_LAYER == 10
    and len(pca_results["Handcrafted"]["test"]) == 228
    and len(pca_results["MERT"]["test"]) == 228
    and len(silhouette_df) == 6
)

print("===== FINAL RESULT =====")
print(
    "Generator Fingerprint Visualization Core QC PASS:",
    core_qc_pass,
)


## 최신 실행 결과 요약 (2026-09-13)

- Strict balanced Test 228 tracks를 Handcrafted와 MERT 공간에서 PCA/UMAP으로 시각화했다.
- 원공간 generator silhouette는 Handcrafted **-0.0183**, MERT **0.0164**로 전역 군집 분리는 약하다.
- MERT의 generator silhouette는 genre silhouette(-0.0188)보다 0.0352 높지만 절대값은 작다.
- 높은 supervised attribution 성능과 낮은 2D/전역 silhouette는 fingerprint가 단순한 compact cluster가 아니라 고차원 비선형 경계에 있음을 시사한다.
- 시각화와 표는 `results/generator_fingerprint_visualization/`에 저장했다.

**최종 상태: Generator Fingerprint Visualization Core QC PASS = True.**
